# 🎬 Movie Dataset — Preprocessing Documentation

> **Datasets:** `tmdb_5000_movies.csv` · `tmdb_5000_credits.csv`  
> **Merged on:** `id` · **Final rows:** ~4,803 movies


## 2 · ❌ Dropped Columns

### Duplicate Titles
| Dropped | Reason |
|---|---|
| `title_x` | Same as `original_title` (movies CSV) |
| `title_y` | Same as `original_title` (credits CSV) |
| `repeat_count` *(temp)* | Helper column to check duplicates — removed after use |

### Redundant / Low-Value
| Dropped | Reason |
|---|---|
| `homepage` | Mostly NaN |
| `status` | 99.8% = "Released" — no useful variance |
| `original_language` | Redundant with `spoken_languages` |
| `id` | Not needed after merge |

### Raw JSON → Replaced by parsed columns
| Dropped | Replaced by |
|---|---|
| `cast` (raw JSON) | `cast_names` + `male_count` + `female_count` |
| `cast_gender` *(intermediate)* | `male_count`, `female_count` |
| `crew` (raw JSON) | `director`, `producers`, `top_jobs` |

### Scale Changed
| Dropped | Replaced by |
|---|---|
| `budget` | `budget_M` (millions) |
| `revenue` | `revenue_M` (millions, 2 decimals) |
| `release_date` | `movie_age` (2026 − release year) |


## 3 · ✅ New Columns (Feature Engineering)

| New Column | Source | How | Example |
|---|---|---|---|
| `cast_names` | `cast` JSON | Extract `name` for each cast member | `"Tom Hanks, Robin Wright, ..."` |
| `male_count` | `cast` JSON | Count gender code `2` (male) | `35` |
| `female_count` | `cast` JSON | Count gender code `1` (female) | `7` |
| `director` | `crew` JSON | Names where `job == "Director"` | `"Robert Zemeckis"` |
| `producers` | `crew` JSON | Names where `job == "Producer"` | `"Wendy Finerman, Steve Tisch"` |
| `top_jobs` | `crew` JSON | Top 5 most frequent crew jobs | `"Greensman, Leadman, Set Designer, ..."` |
| `budget_M` | `budget` | `budget ÷ 1,000,000` | `55.0` |
| `revenue_M` | `revenue` | `revenue ÷ 1,000,000` (rounded 2dp) | `677.95` |
| `movie_age` | `release_date` | `2026 − release year` | `32.0` |


## 4 · 🔄 Transformations Summary

### JSON String → Plain Text  (`ast.literal_eval`)
| Column | Before | After |
|---|---|---|
| `genres` | `[{"id": 18, "name": "Drama"}, ...]` | `"Drama, Comedy"` |
| `keywords` | `[{"id": 1463, "name": "culture clash"}, ...]` | `"culture clash, future, ..."` |
| `production_companies` | `[{"id": 4, "name": "Pixar"}, ...]` | `"Pixar Animation Studios"` |
| `spoken_languages` | `[{"iso_639_1": "en", "name": "English"}]` | `"English"` |
| `production_countries` | `[{"iso_3166_1": "US", "name": "United States..."}, ...]` | `"United States of America"` |

> **Country rule:** If USA appears in list → use it. Otherwise → first country listed.

### Numeric Scaling
| Column | Change |
|---|---|
| `budget` → `budget_M` | ÷ 1,000,000 |
| `revenue` → `revenue_M` | ÷ 1,000,000, rounded 2dp |
| `popularity` | Rounded to 2 decimal places |

### Date → Age
| Column | Change |
|---|---|
| `release_date` → `movie_age` | `pd.to_datetime` → extract year → `2026 − year` |

### Merge
```python
df_credits.rename(columns={'movie_id': 'id'}, inplace=True)
full_dataset = pd.merge(df_movies, df_credits, on='id', how='outer')


In [500]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [501]:
df_credits=pd.read_csv('tmdb_5000_credits.csv')
df_movies=pd.read_csv('tmdb_5000_movies.csv')

In [502]:
df_credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [503]:
df_credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [504]:
df_movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [505]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [506]:
df_movies.isna().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [507]:
df_credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [508]:
df_credits.rename(columns={'movie_id':'id'},inplace=True)

In [509]:
full_dataset=pd.merge(df_movies,df_credits,on='id',how='outer')

In [510]:
full_dataset.duplicated().sum()

0

In [511]:
full_dataset.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title_x,vote_average,vote_count,title_y,cast,crew
0,4000000,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...",NaN,5,"[{""id"": 612, ""name"": ""hotel""}, {""id"": 613, ""na...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-12-09,4300000,98.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Twelve outrageous guests. Four scandalous requ...,Four Rooms,6.5,530,Four Rooms,"[{""cast_id"": 42, ""character"": ""Ted the Bellhop...","[{""credit_id"": ""52fe420dc3a36847f800012d"", ""de..."
1,11000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.starwars.com/films/star-wars-episod...,11,"[{""id"": 803, ""name"": ""android""}, {""id"": 4270, ...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1977-05-25,775398007,121.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"A long time ago in a galaxy far, far away...",Star Wars,8.1,6624,Star Wars,"[{""cast_id"": 3, ""character"": ""Luke Skywalker"",...","[{""credit_id"": ""52fe420dc3a36847f8000437"", ""de..."


In [512]:
full_dataset["spoken_languages"].value_counts()

spoken_languages
[{"iso_639_1": "en", "name": "English"}]                                                                                                                                                                                                                                                                                            3171
[{"iso_639_1": "en", "name": "English"}, {"iso_639_1": "es", "name": "Espa\u00f1ol"}]                                                                                                                                                                                                                                                127
[{"iso_639_1": "en", "name": "English"}, {"iso_639_1": "fr", "name": "Fran\u00e7ais"}]                                                                                                                                                                                                                                               

### `Abstract Syntax Tree= ast`
 - to convert string list into list

In [513]:
import ast

full_dataset["spoken_languages"] = full_dataset["spoken_languages"].apply(
    lambda x: ", ".join([lang["name"] for lang in ast.literal_eval(x)]) if x != "[]" else "None"
)

In [514]:
full_dataset["repeat_count"] = full_dataset[["original_title", "title_x", "title_y"]].nunique(axis=1)

In [515]:
full_dataset["repeat_count"].value_counts()

repeat_count
1    4542
2     261
Name: count, dtype: int64

### `i observe that this columns have same value ,so i will removed two columns (X,Y)`

In [516]:
full_dataset["genres"].value_counts()

genres
[{"id": 18, "name": "Drama"}]                                                                                                                                       370
[{"id": 35, "name": "Comedy"}]                                                                                                                                      282
[{"id": 18, "name": "Drama"}, {"id": 10749, "name": "Romance"}]                                                                                                     164
[{"id": 35, "name": "Comedy"}, {"id": 10749, "name": "Romance"}]                                                                                                    144
[{"id": 35, "name": "Comedy"}, {"id": 18, "name": "Drama"}]                                                                                                         142
                                                                                                                                                         

In [517]:
import ast

full_dataset["genres"] = full_dataset["genres"].apply(
    lambda x: ", ".join([i["name"] for i in ast.literal_eval(x)]) if x != "[]" else "None"
)

In [518]:
full_dataset["keywords"].value_counts()

keywords
[]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        412
[{"id": 10183, "name": "independent film"}]                                                                                                                                                                                                                                                                                                                                                                                      

In [519]:
import ast

full_dataset["keywords"] = full_dataset["keywords"].apply(
    lambda x: ", ".join([i["name"] for i in ast.literal_eval(x)]) if x != "[]" else "None"
)

In [520]:
full_dataset["production_companies"].value_counts()

production_companies
[]                                                                                                                                                                                                                                                                                                                              351
[{"name": "Paramount Pictures", "id": 4}]                                                                                                                                                                                                                                                                                        58
[{"name": "Universal Pictures", "id": 33}]                                                                                                                                                                                                                                                                                       45
[{"name

In [521]:
import ast

full_dataset["production_companies"] = full_dataset["production_companies"].apply(
    lambda x: ", ".join([i["name"] for i in ast.literal_eval(x)]) if x != "[]" else "None"
)

In [522]:
full_dataset["cast"].value_counts()

cast
[]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [523]:
import ast

full_dataset["cast_names"] = full_dataset["cast"].apply(
    lambda x: [i["name"] for i in ast.literal_eval(x)] if x != "[]" else []
)
full_dataset["cast_names"] = full_dataset["cast_names"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)

full_dataset["cast_gender"] = full_dataset["cast"].apply(
    lambda x: [i["gender"] for i in ast.literal_eval(x)] if x != "[]" else []
)
full_dataset["cast_gender"] = full_dataset["cast"].apply(
    lambda x: [i["gender"] for i in ast.literal_eval(x)] if x != "[]" else []
)

In [524]:
full_dataset["crew"].value_counts()

crew
[]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [525]:
import ast
from collections import Counter
full_dataset["director"] = full_dataset["crew"].apply(
    lambda x: [i["name"] for i in ast.literal_eval(x) if i["job"] == "Director"]
)
full_dataset["director"] = full_dataset["director"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)
full_dataset["producers"] = full_dataset["crew"].apply(
    lambda x: [i["name"] for i in ast.literal_eval(x) if i["job"] == "Producer"]
)
full_dataset["producers"] = full_dataset["producers"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)
def top_jobs(x):
    jobs = [i["job"] for i in ast.literal_eval(x)]
    most_common = Counter(jobs).most_common(5)
    return ", ".join([job for job, count in most_common])

full_dataset["top_jobs"] = full_dataset["crew"].apply(top_jobs)

In [526]:
full_dataset.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title_x,vote_average,vote_count,title_y,cast,crew,repeat_count,cast_names,cast_gender,director,producers,top_jobs
0,4000000,"Crime, Comedy",NaN,5,"hotel, new year's eve, witch, bet, hotel room,...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"Miramax Films, A Band Apart","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-12-09,4300000,98.0,English,Released,Twelve outrageous guests. Four scandalous requ...,Four Rooms,6.5,530,Four Rooms,"[{""cast_id"": 42, ""character"": ""Ted the Bellhop...","[{""credit_id"": ""52fe420dc3a36847f800012d"", ""de...",1,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","[2, 2, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 1, 1, 2, ...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra..."
1,11000000,"Adventure, Action, Science Fiction",http://www.starwars.com/films/star-wars-episod...,11,"android, galaxy, hermit, death star, lightsabe...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"Lucasfilm, Twentieth Century Fox Film Corporation","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1977-05-25,775398007,121.0,English,Released,"A long time ago in a galaxy far, far away...",Star Wars,8.1,6624,Star Wars,"[{""cast_id"": 3, ""character"": ""Luke Skywalker"",...","[{""credit_id"": ""52fe420dc3a36847f8000437"", ""de...",1,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...","[2, 2, 1, 2, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 2, ...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire..."
2,94000000,"Animation, Family",http://movies.disney.com/finding-nemo,12,"father son relationship, harbor, underwater, f...",en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.688789,Pixar Animation Studios,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2003-05-30,940335536,100.0,English,Released,"There are 3.7 trillion fish in the ocean, they...",Finding Nemo,7.6,6122,Finding Nemo,"[{""cast_id"": 8, ""character"": ""Marlin (voice)"",...","[{""credit_id"": ""52fe420ec3a36847f80006b1"", ""de...",1,"Albert Brooks, Ellen DeGeneres, Alexander Goul...","[2, 1, 2, 2, 2, 1, 2, 2, 1, 0, 2, 2, 1, 2, 2, ...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,..."
3,55000000,"Comedy, Drama, Romance",NaN,13,"vietnam veteran, hippie, mentally disabled, ru...",en,Forrest Gump,A man with a low IQ has accomplished great thi...,138.133331,Paramount Pictures,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1994-07-06,677945399,142.0,English,Released,"The world will never be the same, once you've ...",Forrest Gump,8.2,7927,Forrest Gump,"[{""cast_id"": 7, ""character"": ""Forrest Gump"", ""...","[{""credit_id"": ""52fe420ec3a36847f800076b"", ""de...",1,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...","[2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2, 0, ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As..."
4,15000000,Drama,http://www.dreamworks.com/ab/,14,"male nudity, female nudity, adultery, midlife ...",en,American Beauty,"Lester Burnham, a depressed suburban father in...",80.878605,"DreamWorks SKG, Jinks/Cohen Company","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1999-09-15,356296601,122.0,English,Released,Look closer.,American Beauty,7.9,3313,American Beauty,"[{""cast_id"": 6, ""character"": ""Lester Burnham"",...","[{""credit_id"": ""52fe420ec3a36847f8000809"", ""de...",1,"Kevin Spacey, Annette Bening, Thora Birch, Wes...","[2, 1, 1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, ...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting"


In [527]:
full_dataset.drop(columns=["title_x", "title_y", "crew", "cast"], inplace=True)

In [528]:
full_dataset.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,repeat_count,cast_names,cast_gender,director,producers,top_jobs
0,4000000,"Crime, Comedy",NaN,5,"hotel, new year's eve, witch, bet, hotel room,...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"Miramax Films, A Band Apart","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-12-09,4300000,98.0,English,Released,Twelve outrageous guests. Four scandalous requ...,6.5,530,1,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","[2, 2, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 1, 1, 2, ...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra..."
1,11000000,"Adventure, Action, Science Fiction",http://www.starwars.com/films/star-wars-episod...,11,"android, galaxy, hermit, death star, lightsabe...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"Lucasfilm, Twentieth Century Fox Film Corporation","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1977-05-25,775398007,121.0,English,Released,"A long time ago in a galaxy far, far away...",8.1,6624,1,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...","[2, 2, 1, 2, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 2, ...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire..."
2,94000000,"Animation, Family",http://movies.disney.com/finding-nemo,12,"father son relationship, harbor, underwater, f...",en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.688789,Pixar Animation Studios,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2003-05-30,940335536,100.0,English,Released,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,1,"Albert Brooks, Ellen DeGeneres, Alexander Goul...","[2, 1, 2, 2, 2, 1, 2, 2, 1, 0, 2, 2, 1, 2, 2, ...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,..."
3,55000000,"Comedy, Drama, Romance",NaN,13,"vietnam veteran, hippie, mentally disabled, ru...",en,Forrest Gump,A man with a low IQ has accomplished great thi...,138.133331,Paramount Pictures,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1994-07-06,677945399,142.0,English,Released,"The world will never be the same, once you've ...",8.2,7927,1,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...","[2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2, 0, ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As..."
4,15000000,Drama,http://www.dreamworks.com/ab/,14,"male nudity, female nudity, adultery, midlife ...",en,American Beauty,"Lester Burnham, a depressed suburban father in...",80.878605,"DreamWorks SKG, Jinks/Cohen Company","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1999-09-15,356296601,122.0,English,Released,Look closer.,7.9,3313,1,"Kevin Spacey, Annette Bening, Thora Birch, Wes...","[2, 1, 1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, ...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting"


In [529]:
full_dataset["male_count"] = full_dataset["cast_gender"].apply(
    lambda x: x.count(2) if isinstance(x, list) else 0
)

full_dataset["female_count"] = full_dataset["cast_gender"].apply(
    lambda x: x.count(1) if isinstance(x, list) else 0
)

full_dataset["production_countries"] = full_dataset["production_countries"].apply(
    lambda x: ", ".join([i["name"] for i in ast.literal_eval(x)]) if x != "[]" else "None"
)
full_dataset.drop(columns=["cast_gender"],inplace=True)

### `this is fake column i was created`

In [530]:
full_dataset["repeat_count"].value_counts()

repeat_count
1    4542
2     261
Name: count, dtype: int64

In [531]:
full_dataset.drop(columns=["repeat_count"],inplace=True)

In [532]:
full_dataset.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast_names,director,producers,top_jobs,male_count,female_count
0,4000000,"Crime, Comedy",NaN,5,"hotel, new year's eve, witch, bet, hotel room,...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"Miramax Films, A Band Apart",United States of America,1995-12-09,4300000,98.0,English,Released,Twelve outrageous guests. Four scandalous requ...,6.5,530,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra...",8,14
1,11000000,"Adventure, Action, Science Fiction",http://www.starwars.com/films/star-wars-episod...,11,"android, galaxy, hermit, death star, lightsabe...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"Lucasfilm, Twentieth Century Fox Film Corporation",United States of America,1977-05-25,775398007,121.0,English,Released,"A long time ago in a galaxy far, far away...",8.1,6624,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire...",46,3
2,94000000,"Animation, Family",http://movies.disney.com/finding-nemo,12,"father son relationship, harbor, underwater, f...",en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.688789,Pixar Animation Studios,United States of America,2003-05-30,940335536,100.0,English,Released,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,"Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,...",16,5
3,55000000,"Comedy, Drama, Romance",NaN,13,"vietnam veteran, hippie, mentally disabled, ru...",en,Forrest Gump,A man with a low IQ has accomplished great thi...,138.133331,Paramount Pictures,United States of America,1994-07-06,677945399,142.0,English,Released,"The world will never be the same, once you've ...",8.2,7927,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As...",35,7
4,15000000,Drama,http://www.dreamworks.com/ab/,14,"male nudity, female nudity, adultery, midlife ...",en,American Beauty,"Lester Burnham, a depressed suburban father in...",80.878605,"DreamWorks SKG, Jinks/Cohen Company",United States of America,1999-09-15,356296601,122.0,English,Released,Look closer.,7.9,3313,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting",15,22


In [533]:
full_dataset["budget_M"] = full_dataset["budget"] / 1_000_000
full_dataset.drop(columns=["budget"],inplace=True)
full_dataset["popularity"] = full_dataset["popularity"].round(2)

In [534]:
full_dataset.head()

,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast_names,director,producers,top_jobs,male_count,female_count,budget_M
0,"Crime, Comedy",NaN,5,"hotel, new year's eve, witch, bet, hotel room,...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.88,"Miramax Films, A Band Apart",United States of America,1995-12-09,4300000,98.0,English,Released,Twelve outrageous guests. Four scandalous requ...,6.5,530,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra...",8,14,4.0
1,"Adventure, Action, Science Fiction",http://www.starwars.com/films/star-wars-episod...,11,"android, galaxy, hermit, death star, lightsabe...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.39,"Lucasfilm, Twentieth Century Fox Film Corporation",United States of America,1977-05-25,775398007,121.0,English,Released,"A long time ago in a galaxy far, far away...",8.1,6624,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire...",46,3,11.0
2,"Animation, Family",http://movies.disney.com/finding-nemo,12,"father son relationship, harbor, underwater, f...",en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.69,Pixar Animation Studios,United States of America,2003-05-30,940335536,100.0,English,Released,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,"Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,...",16,5,94.0
3,"Comedy, Drama, Romance",NaN,13,"vietnam veteran, hippie, mentally disabled, ru...",en,Forrest Gump,A man with a low IQ has accomplished great thi...,138.13,Paramount Pictures,United States of America,1994-07-06,677945399,142.0,English,Released,"The world will never be the same, once you've ...",8.2,7927,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As...",35,7,55.0
4,Drama,http://www.dreamworks.com/ab/,14,"male nudity, female nudity, adultery, midlife ...",en,American Beauty,"Lester Burnham, a depressed suburban father in...",80.88,"DreamWorks SKG, Jinks/Cohen Company",United States of America,1999-09-15,356296601,122.0,English,Released,Look closer.,7.9,3313,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting",15,22,15.0


In [535]:
full_dataset["repeat_count"] = full_dataset[["spoken_languages", "original_language"]].nunique(axis=1)

In [536]:
full_dataset["repeat_count"].value_counts()

repeat_count
2    4803
Name: count, dtype: int64

### `so i will drop one of them`

In [537]:
full_dataset["status"].value_counts()

status
Released           4795
Rumored               5
Post Production       3
Name: count, dtype: int64

In [538]:
full_dataset["revenue_M"] = full_dataset["revenue"] / 1_000_000
full_dataset["revenue_M"] = full_dataset["revenue_M"].round(2)
full_dataset.drop(columns=["original_language","repeat_count","id","homepage","revenue","status"],inplace=True)
full_dataset.head()

,genres,keywords,original_title,overview,popularity,production_companies,production_countries,release_date,runtime,spoken_languages,tagline,vote_average,vote_count,cast_names,director,producers,top_jobs,male_count,female_count,budget_M,revenue_M
0,"Crime, Comedy","hotel, new year's eve, witch, bet, hotel room,...",Four Rooms,It's Ted the Bellhop's first night on the job....,22.88,"Miramax Films, A Band Apart",United States of America,1995-12-09,98.0,English,Twelve outrageous guests. Four scandalous requ...,6.5,530,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra...",8,14,4.0,4.30
1,"Adventure, Action, Science Fiction","android, galaxy, hermit, death star, lightsabe...",Star Wars,Princess Leia is captured and held hostage by ...,126.39,"Lucasfilm, Twentieth Century Fox Film Corporation",United States of America,1977-05-25,121.0,English,"A long time ago in a galaxy far, far away...",8.1,6624,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire...",46,3,11.0,775.40
2,"Animation, Family","father son relationship, harbor, underwater, f...",Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.69,Pixar Animation Studios,United States of America,2003-05-30,100.0,English,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,"Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,...",16,5,94.0,940.34
3,"Comedy, Drama, Romance","vietnam veteran, hippie, mentally disabled, ru...",Forrest Gump,A man with a low IQ has accomplished great thi...,138.13,Paramount Pictures,United States of America,1994-07-06,142.0,English,"The world will never be the same, once you've ...",8.2,7927,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As...",35,7,55.0,677.95
4,Drama,"male nudity, female nudity, adultery, midlife ...",American Beauty,"Lester Burnham, a depressed suburban father in...",80.88,"DreamWorks SKG, Jinks/Cohen Company",United States of America,1999-09-15,122.0,English,Look closer.,7.9,3313,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting",15,22,15.0,356.30


In [539]:
full_dataset["release_date"] = pd.to_datetime(full_dataset["release_date"])
full_dataset["movie_age"] = 2026 - full_dataset["release_date"].dt.year
full_dataset.drop(columns=["release_date"],inplace=True)

In [540]:
full_dataset["production_countries"].value_counts()

production_countries
United States of America                                                                      2977
United Kingdom, United States of America                                                       181
None                                                                                           174
United Kingdom                                                                                 131
Germany, United States of America                                                              119
                                                                                              ... 
New Zealand, Pakistan, United States of America                                                  1
Canada, Russia, United States of America                                                         1
Belgium, France, Germany, Japan, Norway, Romania, United Kingdom, United States of America       1
Switzerland, Germany, Spain, France, Italy                                              

In [541]:
def clean_country(x):
    if x == "None":
        return "None"
    
    countries = [c.strip() for c in x.split(",")]
    
    if "United States of America" in countries:
        return "United States of America"
    else:
        return countries[0]

full_dataset["production_countries"] = full_dataset["production_countries"].apply(clean_country)

In [542]:
full_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   genres                4803 non-null   object 
 1   keywords              4803 non-null   object 
 2   original_title        4803 non-null   object 
 3   overview              4800 non-null   object 
 4   popularity            4803 non-null   float64
 5   production_companies  4803 non-null   object 
 6   production_countries  4803 non-null   object 
 7   runtime               4801 non-null   float64
 8   spoken_languages      4803 non-null   object 
 9   tagline               3959 non-null   object 
 10  vote_average          4803 non-null   float64
 11  vote_count            4803 non-null   int64  
 12  cast_names            4803 non-null   object 
 13  director              4803 non-null   object 
 14  producers             4803 non-null   object 
 15  top_jobs             

In [543]:
full_dataset.head()

,genres,keywords,original_title,overview,popularity,production_companies,production_countries,runtime,spoken_languages,tagline,vote_average,vote_count,cast_names,director,producers,top_jobs,male_count,female_count,budget_M,revenue_M,movie_age
0,"Crime, Comedy","hotel, new year's eve, witch, bet, hotel room,...",Four Rooms,It's Ted the Bellhop's first night on the job....,22.88,"Miramax Films, A Band Apart",United States of America,98.0,English,Twelve outrageous guests. Four scandalous requ...,6.5,530,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra...",8,14,4.0,4.30,31.0
1,"Adventure, Action, Science Fiction","android, galaxy, hermit, death star, lightsabe...",Star Wars,Princess Leia is captured and held hostage by ...,126.39,"Lucasfilm, Twentieth Century Fox Film Corporation",United States of America,121.0,English,"A long time ago in a galaxy far, far away...",8.1,6624,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire...",46,3,11.0,775.40,49.0
2,"Animation, Family","father son relationship, harbor, underwater, f...",Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.69,Pixar Animation Studios,United States of America,100.0,English,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,"Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,...",16,5,94.0,940.34,23.0
3,"Comedy, Drama, Romance","vietnam veteran, hippie, mentally disabled, ru...",Forrest Gump,A man with a low IQ has accomplished great thi...,138.13,Paramount Pictures,United States of America,142.0,English,"The world will never be the same, once you've ...",8.2,7927,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As...",35,7,55.0,677.95,32.0
4,Drama,"male nudity, female nudity, adultery, midlife ...",American Beauty,"Lester Burnham, a depressed suburban father in...",80.88,"DreamWorks SKG, Jinks/Cohen Company",United States of America,122.0,English,Look closer.,7.9,3313,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting",15,22,15.0,356.30,27.0


In [544]:
full_dataset["production_countries"].value_counts()

production_countries
United States of America    3956
None                         174
United Kingdom               163
France                        92
Canada                        88
Germany                       29
Australia                     29
India                         28
China                         24
Spain                         19
Japan                         17
Italy                         13
Ireland                       12
Hong Kong                     12
Belgium                       12
Mexico                        12
Denmark                       11
South Korea                   10
New Zealand                   10
Brazil                         9
Russia                         9
South Africa                   8
Czech Republic                 7
Argentina                      6
Switzerland                    5
Netherlands                    5
Norway                         4
Austria                        3
Thailand                       3
Sweden                

In [545]:
full_dataset["production_companies"].value_counts()

production_companies
None                                                                                                                                             351
Paramount Pictures                                                                                                                                58
Universal Pictures                                                                                                                                45
New Line Cinema                                                                                                                                   38
Columbia Pictures                                                                                                                                 37
                                                                                                                                                ... 
Village Roadshow Pictures, The Zanuck Company, Heyday films, Warner Bros.            

In [546]:
full_dataset["main_company"] = full_dataset["production_companies"].apply(
    lambda x: x.split(",")[0] if x != "None" else "None"
)
full_dataset["keywords"] = full_dataset["keywords"].apply(
    lambda x: ", ".join(x.split(",")[:5])
)
full_dataset.drop(columns=["production_companies"],inplace=True)

In [547]:
full_dataset["profit"] = full_dataset["revenue_M"] - full_dataset["budget_M"]
full_dataset["profit"] = full_dataset["profit"].round(2)
full_dataset["roi"] = full_dataset.apply(
    lambda x: x["revenue_M"] / x["budget_M"] if x["budget_M"] != 0 else 0,
    axis=1
)
full_dataset["roi"] = full_dataset["roi"].round(2)
full_dataset["cast_size"] = full_dataset["male_count"] + full_dataset["female_count"]

In [548]:
full_dataset

,genres,keywords,original_title,overview,popularity,production_countries,runtime,spoken_languages,tagline,vote_average,vote_count,cast_names,director,producers,top_jobs,male_count,female_count,budget_M,revenue_M,movie_age,main_company,profit,roi,cast_size
0,"Crime, Comedy","hotel, new year's eve, witch, bet, hotel room",Four Rooms,It's Ted the Bellhop's first night on the job....,22.88,United States of America,98.0,English,Twelve outrageous guests. Four scandalous requ...,6.5,530,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra...",8,14,4.000000,4.30,31.0,Miramax Films,0.30,1.08,22
1,"Adventure, Action, Science Fiction","android, galaxy, hermit, death star, light...",Star Wars,Princess Leia is captured and held hostage by ...,126.39,United States of America,121.0,English,"A long time ago in a galaxy far, far away...",8.1,6624,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire...",46,3,11.000000,775.40,49.0,Lucasfilm,764.40,70.49,49
2,"Animation, Family","father son relationship, harbor, underwater,...",Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.69,United States of America,100.0,English,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,"Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,...",16,5,94.000000,940.34,23.0,Pixar Animation Studios,846.34,10.00,21
3,"Comedy, Drama, Romance","vietnam veteran, hippie, mentally disabled, ...",Forrest Gump,A man with a low IQ has accomplished great thi...,138.13,United States of America,142.0,English,"The world will never be the same, once you've ...",8.2,7927,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As...",35,7,55.000000,677.95,32.0,Paramount Pictures,622.95,12.33,42
4,Drama,"male nudity, female nudity, adultery, midli...",American Beauty,"Lester Burnham, a depressed suburban father in...",80.88,United States of America,122.0,English,Look closer.,7.9,3313,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting",15,22,15.000000,356.30,27.0,DreamWorks SKG,341.30,23.75,37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,Horror,None,Midnight Cabaret,A Broadway producer puts on a play with a Devi...,0.00,None,94.0,English,The hot spot where Satan's waitin'.,0.0,0,"Lisa Hart Carroll, Michael Des Barres, Paul Dr...",Pece Dingo,,Director,4,3,0.000000,0.00,36.0,None,0.00,0.00,7
4799,"Comedy, Family, Drama",None,Growing Up Smith,"In 1979, an Indian family moves to America wit...",0.71,None,102.0,English,It’s better to stand out than to fit in.,7.4,7,"Roni Akurati, Brighton Sharbino, Jason Lee, An...",Frank Lotito,,"Director, Screenplay, Editor",5,4,0.000000,0.00,9.0,None,0.00,0.00,9
4800,"Thriller, Drama","christian film, sex trafficking",8 Days,"After sneaking to a party with her friends, 16...",0.02,United States of America,90.0,English,She never knew it could happen to her...,0.0,0,"Nicole Smolen, Kim Baldwin, Ariana Stephens, B...",Jaco Booyens,,"Writer, Director",0,0,0.000000,0.00,12.0,After Eden Pictures,0.00,0.00,0
4801,Family,None,Running Forever,After being estranged since her mother's death...,0.03,United States of America,88.0,None,NaN,0.0,0,,,,,0,0,0.000000,0.00,11.0,New Kingdom Pictures,0.00,0.00,0


In [ ]:
full_dataset["cast_size"][full_dataset["cast_size"]== 0].value_counts()

cast_size
0    86
Name: count, dtype: int64

In [550]:
full_dataset.isna().sum()

genres                    0
keywords                  0
original_title            0
overview                  3
popularity                0
production_countries      0
runtime                   2
spoken_languages          0
tagline                 844
vote_average              0
vote_count                0
cast_names                0
director                  0
producers                 0
top_jobs                  0
male_count                0
female_count              0
budget_M                  0
revenue_M                 0
movie_age                 1
main_company              0
profit                    0
roi                       0
cast_size                 0
dtype: int64

In [557]:
full_dataset["tagline"] = full_dataset["tagline"].replace("None", "Unknown").fillna("Unknown")

In [558]:
full_dataset.isna().sum()

genres                  0
keywords                0
original_title          0
overview                3
popularity              0
production_countries    0
runtime                 2
spoken_languages        0
tagline                 0
vote_average            0
vote_count              0
cast_names              0
director                0
producers               0
top_jobs                0
male_count              0
female_count            0
budget_M                0
revenue_M               0
movie_age               1
main_company            0
profit                  0
roi                     0
cast_size               0
dtype: int64

In [560]:
full_dataset.dropna(inplace=True)

In [561]:
full_dataset.head()

,genres,keywords,original_title,overview,popularity,production_countries,runtime,spoken_languages,tagline,vote_average,vote_count,cast_names,director,producers,top_jobs,male_count,female_count,budget_M,revenue_M,movie_age,main_company,profit,roi,cast_size
0,"Crime, Comedy","hotel, new year's eve, witch, bet, hotel room",Four Rooms,It's Ted the Bellhop's first night on the job....,22.88,United States of America,98.0,English,Twelve outrageous guests. Four scandalous requ...,6.5,530,"Tim Roth, Antonio Banderas, Jennifer Beals, Ma...","Allison Anders, Alexandre Rockwell, Robert Rod...",Lawrence Bender,"Director, Writer, Editor, Director of Photogra...",8,14,4.0,4.30,31.0,Miramax Films,0.30,1.08,22
1,"Adventure, Action, Science Fiction","android, galaxy, hermit, death star, light...",Star Wars,Princess Leia is captured and held hostage by ...,126.39,United States of America,121.0,English,"A long time ago in a galaxy far, far away...",8.1,6624,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,"Gary Kurtz, Rick McCallum","Casting, Editor, Producer, Art Direction, Dire...",46,3,11.0,775.40,49.0,Lucasfilm,764.40,70.49,49
2,"Animation, Family","father son relationship, harbor, underwater,...",Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.69,United States of America,100.0,English,"There are 3.7 trillion fish in the ocean, they...",7.6,6122,"Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,Graham Walters,"Visual Development, CG Supervisor, CG Painter,...",16,5,94.0,940.34,23.0,Pixar Animation Studios,846.34,10.00,21
3,"Comedy, Drama, Romance","vietnam veteran, hippie, mentally disabled, ...",Forrest Gump,A man with a low IQ has accomplished great thi...,138.13,United States of America,142.0,English,"The world will never be the same, once you've ...",8.2,7927,"Tom Hanks, Robin Wright, Gary Sinise, Mykelti ...",Robert Zemeckis,"Wendy Finerman, Steve Tisch, Steve Starkey","Greensman, Leadman, Set Designer, Producer, As...",35,7,55.0,677.95,32.0,Paramount Pictures,622.95,12.33,42
4,Drama,"male nudity, female nudity, adultery, midli...",American Beauty,"Lester Burnham, a depressed suburban father in...",80.88,United States of America,122.0,English,Look closer.,7.9,3313,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"Bruce Cohen, Dan Jinks","Foley, Co-Producer, Producer, Editor, Casting",15,22,15.0,356.30,27.0,DreamWorks SKG,341.30,23.75,37


In [ ]:
full_dataset.to_csv("Movies_Clean.csv")